# LG-CAFN-R9 and LG-CAFN-R17 training

This notebook is the self-contained training notebook for the BreastDM faithful reconstruction.

- **LG-CAFN-R9** uses the nine-sequence ROI arrays.
- **LG-CAFN-R17** uses the seventeen-sequence ROI arrays.
- **R** means reconstruction because the paper does not disclose how its 9/17-channel arrays were converted to the three-channel inputs expected by the released ImageNet models.
- The canonical patient split is 166 training, 19 validation, and 47 testing patients.
- The six erroneous \`img9Se/test/Malignant\` assignments and their 43 files must already have been removed from \`BreaDM_cleaned.zip\`.

Run this notebook with a Colab GPU runtime. Full training is disabled initially; all validation and diagnostic cells should pass before setting \`RUN_FULL_TRAINING = True\`.


In [ ]:
# Cell 1: install packages, mount Drive, and define paths
!pip install -q timm scikit-learn pandas matplotlib

from google.colab import drive
from pathlib import Path
from collections import defaultdict
from io import BytesIO
import copy
import csv
import json
import math
import os
import random
import sys
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE_ROOT / "LG_CAFN_Reproduction"
DATA_DIR = PROJECT_DIR / "data"
MANIFEST_DIR = PROJECT_DIR / "manifests"
STATS_DIR = PROJECT_DIR / "preprocessing_statistics"
EXPERIMENTS_DIR = PROJECT_DIR / "experiments"
CONFIG_DIR = PROJECT_DIR / "configs"

ZIP_PATH = DATA_DIR / "BreaDM_cleaned.zip"

for directory in (STATS_DIR, EXPERIMENTS_DIR, CONFIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

REQUIRED_FILES = [
    ZIP_PATH,
    MANIFEST_DIR / "img9Se_manifest.csv",
    MANIFEST_DIR / "img17Se_manifest.csv",
]

missing = [path for path in REQUIRED_FILES if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required artifacts:\n" + "\n".join(map(str, missing)))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("Enable a Colab GPU runtime before continuing.")

print("Project:", PROJECT_DIR)
print("GPU:", torch.cuda.get_device_name(0))
print("Artifacts: PASS")


In [ ]:
# Cell 2: experiment configuration and reproducibility
SEED = 8
RESIZE_SIZE = 256
CROP_SIZE = 224
EFFECTIVE_BATCH_SIZE = 32
NUM_WORKERS = 2
MAX_EPOCHS = 100
INITIAL_LR = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 0.01
LR_PATIENCE = 10
EARLY_STOPPING_PATIENCE = 20

BRANCH_CHANNELS = {"img9Se": 9, "img17Se": 17}
LABEL_TO_INDEX = {"Benign": 0, "Malignant": 1}

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)

seed_everything()

protocol = {
    "name": "LG-CAFN faithful reconstruction",
    "experiments": {"LG_CAFN_R9": "img9Se", "LG_CAFN_R17": "img17Se"},
    "canonical_patient_counts": {"train": 166, "val": 19, "test": 47},
    "canonical_sample_counts": {"train": 1202, "val": 117, "test": 403},
    "seed": SEED,
    "resize": RESIZE_SIZE,
    "crop": CROP_SIZE,
    "effective_batch_size": EFFECTIVE_BATCH_SIZE,
    "optimizer": "SGD",
    "initial_lr": INITIAL_LR,
    "momentum": MOMENTUM,
    "weight_decay": WEIGHT_DECAY,
    "maximum_epochs": MAX_EPOCHS,
    "checkpoint_metric": "validation patient-level ROC AUC",
    "reconstruction_decisions": [
        "training-only per-sequence normalization",
        "learned 1x1 temporal projection from 9/17 channels to 3",
        "deterministic evaluation resize and center crop",
        "patient probability equals mean ROI malignant probability",
    ],
}

with open(CONFIG_DIR / "reconstruction_protocol.json", "w", encoding="utf-8") as file:
    json.dump(protocol, file, indent=2)

print(json.dumps(protocol, indent=2))


In [ ]:
# Cell 3: complete ZIP-backed dataset and preprocessing implementation
def calculate_training_statistics(branch):
    channels = BRANCH_CHANNELS[branch]
    manifest = pd.read_csv(MANIFEST_DIR / f"{branch}_manifest.csv")
    training = manifest[manifest["split"] == "train"]
    channel_sum = np.zeros(channels, dtype=np.float64)
    channel_square_sum = np.zeros(channels, dtype=np.float64)
    pixel_count = np.zeros(channels, dtype=np.int64)

    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        for row in training.itertuples(index=False):
            array = np.load(BytesIO(archive.read(row.archive_path)), allow_pickle=False)
            if array.ndim != 3 or array.shape[-1] != channels or array.dtype != np.uint8:
                raise ValueError(f"Invalid array at {row.archive_path}: {array.shape}, {array.dtype}")
            array = array.astype(np.float64) / 255.0
            pixels = array.shape[0] * array.shape[1]
            channel_sum += array.sum(axis=(0, 1))
            channel_square_sum += np.square(array).sum(axis=(0, 1))
            pixel_count += pixels

    mean = channel_sum / pixel_count
    variance = np.maximum(channel_square_sum / pixel_count - np.square(mean), 0)
    std = np.sqrt(variance)
    if np.any(std < 1e-6):
        raise ValueError(f"Near-zero channel standard deviation in {branch}.")
    return {
        "mean": mean.tolist(),
        "std": std.tolist(),
        "training_samples": int(len(training)),
        "statistics_partition": "train only",
    }

def load_or_create_statistics(branch):
    path = STATS_DIR / f"{branch}_training_channel_stats.json"
    if path.exists():
        with open(path, "r", encoding="utf-8") as file:
            statistics = json.load(file)
    else:
        statistics = calculate_training_statistics(branch)
        with open(path, "w", encoding="utf-8") as file:
            json.dump(statistics, file, indent=2)
    expected = BRANCH_CHANNELS[branch]
    if len(statistics["mean"]) != expected or len(statistics["std"]) != expected:
        raise ValueError(f"Invalid saved statistics for {branch}.")
    return statistics

class LGCAFNPreprocessor:
    def __init__(self, mean, std, training):
        self.training = bool(training)
        self.mean = torch.tensor(mean, dtype=torch.float32).view(-1, 1, 1)
        self.std = torch.tensor(std, dtype=torch.float32).view(-1, 1, 1)

    def __call__(self, array):
        tensor = torch.from_numpy(np.ascontiguousarray(array)).permute(2, 0, 1).float() / 255.0
        tensor = F.interpolate(
            tensor.unsqueeze(0),
            size=(RESIZE_SIZE, RESIZE_SIZE),
            mode="bilinear",
            align_corners=False,
        ).squeeze(0)

        if self.training:
            maximum = RESIZE_SIZE - CROP_SIZE
            top = random.randint(0, maximum)
            left = random.randint(0, maximum)
            tensor = tensor[:, top:top + CROP_SIZE, left:left + CROP_SIZE]
            if random.random() < 0.5:
                tensor = torch.flip(tensor, dims=(2,))
            if random.random() < 0.5:
                tensor = torch.flip(tensor, dims=(1,))
        else:
            offset = (RESIZE_SIZE - CROP_SIZE) // 2
            tensor = tensor[:, offset:offset + CROP_SIZE, offset:offset + CROP_SIZE]

        tensor = (tensor - self.mean) / self.std
        if not torch.isfinite(tensor).all():
            raise ValueError("Preprocessing produced a non-finite tensor.")
        return tensor.contiguous()

class BreastDMLGCAFNDataset(Dataset):
    def __init__(self, branch, split, transform):
        if branch not in BRANCH_CHANNELS or split not in {"train", "val", "test"}:
            raise ValueError(f"Invalid branch/split: {branch}/{split}")
        self.branch = branch
        self.split = split
        self.channels = BRANCH_CHANNELS[branch]
        self.transform = transform
        self._archive = None
        frame = pd.read_csv(MANIFEST_DIR / f"{branch}_manifest.csv")
        frame = frame[frame["split"] == split].copy()
        frame = frame.sort_values(["label_index", "patient_id", "filename"]).reset_index(drop=True)
        if frame.empty:
            raise ValueError(f"No records for {branch}/{split}.")
        self.records = frame.to_dict("records")

    def _get_archive(self):
        if self._archive is None:
            self._archive = zipfile.ZipFile(ZIP_PATH, "r")
        return self._archive

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        array = np.load(
            BytesIO(self._get_archive().read(record["archive_path"])),
            allow_pickle=False,
        )
        if array.ndim != 3 or array.shape[-1] != self.channels:
            raise ValueError(f"Invalid shape at {record['archive_path']}: {array.shape}")
        return {
            "image": self.transform(array),
            "label": torch.tensor(int(record["label_index"]), dtype=torch.long),
            "patient_id": str(record["patient_id"]),
            "sample_id": str(record["sample_id"]),
            "archive_path": str(record["archive_path"]),
        }

    def __getstate__(self):
        state = self.__dict__.copy()
        state["_archive"] = None
        return state

    def __del__(self):
        if getattr(self, "_archive", None) is not None:
            self._archive.close()

def build_lgcafn_dataloaders(branch, physical_batch_size, num_workers=NUM_WORKERS):
    statistics = load_or_create_statistics(branch)
    train_transform = LGCAFNPreprocessor(statistics["mean"], statistics["std"], True)
    eval_transform = LGCAFNPreprocessor(statistics["mean"], statistics["std"], False)
    datasets = {
        "train": BreastDMLGCAFNDataset(branch, "train", train_transform),
        "val": BreastDMLGCAFNDataset(branch, "val", eval_transform),
        "test": BreastDMLGCAFNDataset(branch, "test", eval_transform),
    }
    generator = torch.Generator().manual_seed(SEED)
    common = dict(
        batch_size=physical_batch_size,
        num_workers=num_workers,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=generator,
        persistent_workers=num_workers > 0,
    )
    loaders = {
        "train": DataLoader(datasets["train"], shuffle=True, drop_last=True, **common),
        "val": DataLoader(datasets["val"], shuffle=False, drop_last=False, **common),
        "test": DataLoader(datasets["test"], shuffle=False, drop_last=False, **common),
    }
    return {"datasets": datasets, "loaders": loaders, "statistics": statistics}

for branch in BRANCH_CHANNELS:
    probe = build_lgcafn_dataloaders(branch, physical_batch_size=2, num_workers=0)
    batch = next(iter(probe["loaders"]["train"]))
    assert tuple(batch["image"].shape) == (2, BRANCH_CHANNELS[branch], 224, 224)
    print(branch, {split: len(probe["datasets"][split]) for split in ("train", "val", "test")})
print("Dataset implementation: PASS")


In [ ]:
# Cell 4: LG-CAFN-R architecture
class TemporalChannelAdapter(nn.Module):
    def __init__(self, input_channels):
        super().__init__()
        self.input_channels = input_channels
        self.projection = nn.Conv2d(input_channels, 3, kernel_size=1, bias=True)
        self.normalization = nn.BatchNorm2d(3)
        self.activation = nn.ReLU(inplace=True)
        with torch.no_grad():
            self.projection.weight.fill_(1.0 / input_channels)
            self.projection.weight.add_(1e-3 * torch.randn_like(self.projection.weight))
            self.projection.bias.zero_()

    def forward(self, x):
        return self.activation(self.normalization(self.projection(x)))

class BidirectionalFeatureCouplingUnit(nn.Module):
    def __init__(self, cnn_channels, transformer_dimension=768, fusion_dimension=256, heads=8):
        super().__init__()
        self.fusion_dimension = fusion_dimension
        self.cnn_to_fusion = nn.Conv2d(cnn_channels, fusion_dimension, 1, bias=False)
        self.token_to_fusion = nn.Linear(transformer_dimension, fusion_dimension, bias=False)
        self.cnn_to_token = nn.MultiheadAttention(fusion_dimension, heads, batch_first=True)
        self.token_to_cnn = nn.MultiheadAttention(fusion_dimension, heads, batch_first=True)
        self.fusion_to_token = nn.Linear(fusion_dimension, transformer_dimension, bias=False)
        self.fusion_to_cnn = nn.Conv2d(fusion_dimension, cnn_channels, 1, bias=False)
        self.token_norm = nn.LayerNorm(transformer_dimension)
        self.cnn_norm = nn.BatchNorm2d(cnn_channels)
        self.token_scale = nn.Parameter(torch.tensor(1e-3))
        self.cnn_scale = nn.Parameter(torch.tensor(1e-3))

    def forward(self, cnn_feature, tokens):
        batch, _, height, width = cnn_feature.shape
        class_token, patch_tokens = tokens[:, :1], tokens[:, 1:]
        cnn_sequence = self.cnn_to_fusion(cnn_feature).flatten(2).transpose(1, 2)
        token_sequence = self.token_to_fusion(patch_tokens)
        token_update, _ = self.cnn_to_token(
            token_sequence, cnn_sequence, cnn_sequence, need_weights=False
        )
        patch_tokens = self.token_norm(
            patch_tokens + self.token_scale * self.fusion_to_token(token_update)
        )
        token_sequence = self.token_to_fusion(patch_tokens)
        cnn_update, _ = self.token_to_cnn(
            cnn_sequence, token_sequence, token_sequence, need_weights=False
        )
        cnn_update = cnn_update.transpose(1, 2).reshape(
            batch, self.fusion_dimension, height, width
        )
        cnn_feature = self.cnn_norm(
            cnn_feature + self.cnn_scale * self.fusion_to_cnn(cnn_update)
        )
        return cnn_feature, torch.cat([class_token, patch_tokens], dim=1)

class LGCAFNReconstruction(nn.Module):
    def __init__(self, input_channels, pretrained=True, classes=2):
        super().__init__()
        self.temporal_adapter = TemporalChannelAdapter(input_channels)
        self.local_branch = timm.create_model(
            "seresnet50",
            pretrained=pretrained,
            features_only=True,
            out_indices=(1, 2, 3, 4),
        )
        local_channels = self.local_branch.feature_info.channels()
        vit = timm.create_model(
            "vit_base_patch16_224",
            pretrained=pretrained,
            num_classes=0,
        )
        self.patch_embed = vit.patch_embed
        self.class_token = vit.cls_token
        self.position_embedding = vit.pos_embed
        self.position_dropout = vit.pos_drop
        self.patch_dropout = vit.patch_drop
        self.pre_norm = vit.norm_pre
        self.transformer_blocks = nn.ModuleList(list(vit.blocks[:7]))
        self.transformer_norm = vit.norm
        self.blocks_per_stage = (2, 2, 2, 1)
        self.coupling_units = nn.ModuleList([
            BidirectionalFeatureCouplingUnit(channels) for channels in local_channels
        ])
        self.local_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(local_channels[-1] + vit.embed_dim),
            nn.Dropout(0.5),
            nn.Linear(local_channels[-1] + vit.embed_dim, classes),
        )

    def _tokens(self, image):
        patches = self.patch_embed(image)
        cls = self.class_token.expand(image.shape[0], -1, -1)
        tokens = torch.cat([cls, patches], dim=1) + self.position_embedding
        return self.pre_norm(self.patch_dropout(self.position_dropout(tokens)))

    def forward(self, x):
        image = self.temporal_adapter(x)
        local_features = self.local_branch(image)
        tokens = self._tokens(image)
        block_index = 0
        fused_local = None
        for stage, block_count in enumerate(self.blocks_per_stage):
            for _ in range(block_count):
                tokens = self.transformer_blocks[block_index](tokens)
                block_index += 1
            fused_local, tokens = self.coupling_units[stage](local_features[stage], tokens)
        tokens = self.transformer_norm(tokens)
        local_vector = self.local_pool(fused_local).flatten(1)
        global_vector = tokens[:, 0]
        return self.classifier(torch.cat([local_vector, global_vector], dim=1))

for branch, channels in BRANCH_CHANNELS.items():
    data = build_lgcafn_dataloaders(branch, physical_batch_size=2, num_workers=0)
    batch = next(iter(data["loaders"]["train"]))
    model = LGCAFNReconstruction(channels, pretrained=True).to(DEVICE)
    images = batch["image"].to(DEVICE)
    labels = batch["label"].to(DEVICE)
    logits = model(images)
    loss = F.cross_entropy(logits, labels)
    loss.backward()
    assert tuple(logits.shape) == (2, 2) and torch.isfinite(loss)
    print(branch, "logits", tuple(logits.shape), "loss", float(loss))
    del model, logits, loss
    torch.cuda.empty_cache()
print("Model forward/backward validation: PASS")


In [ ]:
# Cell 5: balanced overfitting diagnostic and GPU batch-size probe
def balanced_subset_indices(dataset, per_class=8):
    indices = {0: [], 1: []}
    for index, record in enumerate(dataset.records):
        label = int(record["label_index"])
        if len(indices[label]) < per_class:
            indices[label].append(index)
        if all(len(values) == per_class for values in indices.values()):
            break
    if not all(len(values) == per_class for values in indices.values()):
        raise RuntimeError("Could not form a balanced diagnostic subset.")
    return indices[0] + indices[1]

def balanced_overfit_test(branch="img9Se", physical_batch_size=2, epochs=20):
    channels = BRANCH_CHANNELS[branch]
    data = build_lgcafn_dataloaders(
        branch, physical_batch_size=physical_batch_size, num_workers=0
    )
    dataset = data["datasets"]["train"]
    subset = Subset(dataset, balanced_subset_indices(dataset, per_class=8))
    loader = DataLoader(
        subset, batch_size=physical_batch_size, shuffle=True, num_workers=0
    )
    model = LGCAFNReconstruction(channels, pretrained=True).to(DEVICE)
    optimizer = torch.optim.SGD(
        model.parameters(), lr=INITIAL_LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY
    )
    final_accuracy = 0.0
    for epoch in range(1, epochs + 1):
        model.train()
        correct = total = 0
        total_loss = 0.0
        for batch in loader:
            images = batch["image"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
            optimizer.step()
            total_loss += float(loss) * len(labels)
            correct += int((logits.argmax(1) == labels).sum())
            total += len(labels)
        final_accuracy = correct / total
        print(f"epoch={epoch:02d} loss={total_loss/total:.5f} accuracy={final_accuracy:.2%}")
        if final_accuracy >= 0.95:
            break
    if final_accuracy < 0.95:
        raise RuntimeError("Balanced overfitting diagnostic did not reach 95%.")
    del model
    torch.cuda.empty_cache()
    print("Balanced overfitting diagnostic: PASS")

def find_safe_batch_size(branch="img9Se", candidates=(2, 4, 8, 16, 32)):
    channels = BRANCH_CHANNELS[branch]
    safe = None
    for batch_size in candidates:
        torch.cuda.empty_cache()
        try:
            data = build_lgcafn_dataloaders(branch, batch_size, num_workers=0)
            batch = next(iter(data["loaders"]["train"]))
            model = LGCAFNReconstruction(channels, pretrained=True).to(DEVICE)
            images = batch["image"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            loss = F.cross_entropy(model(images), labels)
            loss.backward()
            safe = batch_size
            print("batch", batch_size, "PASS")
            del model, images, labels, loss
        except torch.cuda.OutOfMemoryError:
            print("batch", batch_size, "OOM")
            torch.cuda.empty_cache()
            break
    if safe is None:
        raise RuntimeError("No tested batch size fits GPU memory.")
    return safe

SAFE_BATCH_SIZE = find_safe_batch_size("img9Se")
GRADIENT_ACCUMULATION_STEPS = math.ceil(EFFECTIVE_BATCH_SIZE / SAFE_BATCH_SIZE)
print("Safe physical batch:", SAFE_BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch:", SAFE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)

# Run the balanced diagnostic only after identifying a safe physical batch.
balanced_overfit_test("img9Se", physical_batch_size=SAFE_BATCH_SIZE)


In [ ]:
# Cell 6: metrics, patient aggregation, evaluation, and training engine
def binary_metrics(labels, probabilities, threshold=0.5):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "sensitivity": recall_score(labels, predictions, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        "auc": roc_auc_score(labels, probabilities),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def aggregate_patients(labels, probabilities, patient_ids):
    grouped = defaultdict(lambda: {"labels": [], "probabilities": []})
    for label, probability, patient_id in zip(labels, probabilities, patient_ids):
        grouped[str(patient_id)]["labels"].append(int(label))
        grouped[str(patient_id)]["probabilities"].append(float(probability))
    rows = []
    for patient_id, values in sorted(grouped.items()):
        if len(set(values["labels"])) != 1:
            raise ValueError(f"Conflicting labels for {patient_id}.")
        rows.append({
            "patient_id": patient_id,
            "label": values["labels"][0],
            "probability": float(np.mean(values["probabilities"])),
            "roi_count": len(values["probabilities"]),
        })
    return pd.DataFrame(rows)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    labels, probabilities, patient_ids, sample_ids = [], [], [], []
    total_loss = 0.0
    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        targets = batch["label"].to(DEVICE, non_blocking=True)
        logits = model(images)
        loss = F.cross_entropy(logits, targets)
        probs = torch.softmax(logits, dim=1)[:, 1]
        total_loss += float(loss) * len(targets)
        labels.extend(targets.cpu().tolist())
        probabilities.extend(probs.cpu().tolist())
        patient_ids.extend(batch["patient_id"])
        sample_ids.extend(batch["sample_id"])
    roi_metrics = binary_metrics(labels, probabilities)
    patient_frame = aggregate_patients(labels, probabilities, patient_ids)
    patient_metrics = binary_metrics(patient_frame["label"], patient_frame["probability"])
    predictions = pd.DataFrame({
        "sample_id": sample_ids,
        "patient_id": patient_ids,
        "label": labels,
        "malignant_probability": probabilities,
    })
    return {
        "loss": total_loss / len(labels),
        "roi_metrics": roi_metrics,
        "patient_metrics": patient_metrics,
        "roi_predictions": predictions,
        "patient_predictions": patient_frame,
    }

def train_experiment(experiment_name, branch, physical_batch_size, accumulation_steps):
    output_dir = EXPERIMENTS_DIR / experiment_name
    output_dir.mkdir(parents=True, exist_ok=True)
    data = build_lgcafn_dataloaders(branch, physical_batch_size, NUM_WORKERS)
    model = LGCAFNReconstruction(BRANCH_CHANNELS[branch], pretrained=True).to(DEVICE)
    optimizer = torch.optim.SGD(
        model.parameters(), lr=INITIAL_LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.1, patience=LR_PATIENCE
    )
    best_auc = -np.inf
    epochs_without_improvement = 0
    history = []
    checkpoint_path = output_dir / "best_validation_auc.pt"

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss = correct = examples = 0
        for step, batch in enumerate(data["loaders"]["train"], start=1):
            images = batch["image"].to(DEVICE, non_blocking=True)
            labels = batch["label"].to(DEVICE, non_blocking=True)
            logits = model(images)
            unscaled_loss = F.cross_entropy(logits, labels)
            loss = unscaled_loss / accumulation_steps
            loss.backward()
            if step % accumulation_steps == 0 or step == len(data["loaders"]["train"]):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
            total_loss += float(unscaled_loss) * len(labels)
            correct += int((logits.argmax(1) == labels).sum())
            examples += len(labels)

        validation = evaluate(model, data["loaders"]["val"])
        validation_auc = validation["patient_metrics"]["auc"]
        scheduler.step(validation_auc)
        row = {
            "epoch": epoch,
            "train_loss": total_loss / examples,
            "train_accuracy": correct / examples,
            "val_loss": validation["loss"],
            "val_patient_auc": validation_auc,
            "val_patient_accuracy": validation["patient_metrics"]["accuracy"],
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
        history.append(row)
        print(
            f"{experiment_name} epoch={epoch:03d} "
            f"train_loss={row['train_loss']:.5f} "
            f"train_acc={row['train_accuracy']:.2%} "
            f"val_auc={validation_auc:.5f}"
        )

        if validation_auc > best_auc:
            best_auc = validation_auc
            epochs_without_improvement = 0
            torch.save({
                "model_state": model.state_dict(),
                "epoch": epoch,
                "validation_patient_auc": best_auc,
                "branch": branch,
                "protocol": protocol,
            }, checkpoint_path)
        else:
            epochs_without_improvement += 1

        pd.DataFrame(history).to_csv(output_dir / "history.csv", index=False)
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping.")
            break

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    validation = evaluate(model, data["loaders"]["val"])
    test = evaluate(model, data["loaders"]["test"])
    validation["roi_predictions"].to_csv(output_dir / "validation_roi_predictions.csv", index=False)
    validation["patient_predictions"].to_csv(output_dir / "validation_patient_predictions.csv", index=False)
    test["roi_predictions"].to_csv(output_dir / "test_roi_predictions.csv", index=False)
    test["patient_predictions"].to_csv(output_dir / "test_patient_predictions.csv", index=False)
    summary = {
        "experiment": experiment_name,
        "branch": branch,
        "best_epoch": int(checkpoint["epoch"]),
        "validation_roi_metrics": validation["roi_metrics"],
        "validation_patient_metrics": validation["patient_metrics"],
        "test_roi_metrics": test["roi_metrics"],
        "test_patient_metrics": test["patient_metrics"],
    }
    with open(output_dir / "metrics.json", "w", encoding="utf-8") as file:
        json.dump(summary, file, indent=2)
    return model, summary, pd.DataFrame(history)

print("Training engine defined.")


In [ ]:
# Cell 7: full R9 and R17 training
# Keep False until all earlier cells pass. Change to True to begin the two full experiments.
RUN_FULL_TRAINING = False

if RUN_FULL_TRAINING:
    results = {}
    for experiment_name, branch in (
        ("LG_CAFN_R9", "img9Se"),
        ("LG_CAFN_R17", "img17Se"),
    ):
        model, summary, history = train_experiment(
            experiment_name=experiment_name,
            branch=branch,
            physical_batch_size=SAFE_BATCH_SIZE,
            accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        )
        results[experiment_name] = summary
        print(json.dumps(summary, indent=2))
        del model
        torch.cuda.empty_cache()
else:
    print("Full training is disabled.")
    print("After all diagnostics pass, set RUN_FULL_TRAINING = True and rerun this cell.")


In [ ]:
# Cell 8: load and compare completed experiment results
rows = []
for experiment_name in ("LG_CAFN_R9", "LG_CAFN_R17"):
    metrics_path = EXPERIMENTS_DIR / experiment_name / "metrics.json"
    if not metrics_path.exists():
        print("Not trained yet:", experiment_name)
        continue
    with open(metrics_path, "r", encoding="utf-8") as file:
        metrics = json.load(file)
    patient = metrics["test_patient_metrics"]
    rows.append({
        "experiment": experiment_name,
        "accuracy": patient["accuracy"],
        "precision": patient["precision"],
        "sensitivity": patient["sensitivity"],
        "specificity": patient["specificity"],
        "auc": patient["auc"],
    })

comparison = pd.DataFrame(rows)
display(comparison)
if not comparison.empty:
    comparison.to_csv(EXPERIMENTS_DIR / "LG_CAFN_R9_R17_comparison.csv", index=False)
